# AeroSentinel — Drone Telemetry Analysis & ML Pipeline

**Purpose:** A reproducible notebook for exploring PX4/INAV telemetry, validating processed data, building an Isolation Forest anomaly-detection baseline, and evaluating remaining-flight-time predictions where suitable labels exist.

> Run from the AeroSentinel project root. This notebook does not invent labels or claim results that have not been measured. It is designed to work with the project's local `data/` directory and reports missing inputs clearly.

## Project workflow
1. Configure paths and inspect available data.
2. Load processed telemetry and inspect schema/quality.
3. Explore sensor distributions and flight trajectories.
4. Prepare numeric features and train an Isolation Forest baseline.
5. Evaluate a remaining-flight-time target only if a valid target column is present.
6. Export figures and metrics for the technical report.

**Important:** Keep raw data, train/validation/test splits, and preprocessing leakage-safe. Do not fit scalers or models using test data.


## 1. Environment and configuration

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42

# Run this notebook from the AeroSentinel project root.
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORT_DIR = PROJECT_ROOT / "reports" / "notebook"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Processed data exists:", PROCESSED_DIR.exists())
print("Report output:", REPORT_DIR.resolve())


## 2. Discover available datasets

In [ ]:
if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"Cannot find {PROCESSED_DIR}. Open Jupyter with the AeroSentinel project root as the working directory."
    )

csv_files = sorted(PROCESSED_DIR.rglob("*.csv"))
parquet_files = sorted(PROCESSED_DIR.rglob("*.parquet"))

print(f"CSV files: {len(csv_files)}")
print(f"Parquet files: {len(parquet_files)}")

inventory = [
    {"path": str(p.relative_to(PROJECT_ROOT)), "format": p.suffix.lower(),
     "size_mb": round(p.stat().st_size / (1024 * 1024), 3)}
    for p in csv_files + parquet_files
]
inventory_df = pd.DataFrame(inventory)
display(inventory_df.head(30))
inventory_df.to_csv(REPORT_DIR / "data_inventory.csv", index=False)


## 3. Select and load a telemetry file

The notebook prefers Parquet when available, otherwise CSV. Choose a file from the inventory. For large files, use the optional row limit for an initial inspection.

In [ ]:
# Set this to a path shown in the inventory, or leave None to select the first available file.
SELECTED_FILE = None
ROW_LIMIT = None  # e.g. 500_000 for a quick inspection; None loads the full file

available_files = parquet_files + csv_files
if not available_files:
    raise FileNotFoundError("No processed CSV or Parquet files were found.")

if SELECTED_FILE is None:
    selected_path = available_files[0]
else:
    selected_path = (PROJECT_ROOT / SELECTED_FILE).resolve()
    if not selected_path.exists():
        raise FileNotFoundError(f"Selected file does not exist: {selected_path}")

print("Selected:", selected_path)

if selected_path.suffix.lower() == ".parquet":
    df = pd.read_parquet(selected_path)
else:
    df = pd.read_csv(selected_path, nrows=ROW_LIMIT)

print("Shape:", df.shape)
display(df.head())


## 4. Schema and data-quality audit

In [ ]:
schema_df = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(x) for x in df.dtypes],
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_pct": [round(float(df[c].isna().mean() * 100), 2) for c in df.columns],
    "unique_count": [int(df[c].nunique(dropna=True)) for c in df.columns],
})
display(schema_df)
schema_df.to_csv(REPORT_DIR / "selected_file_schema_audit.csv", index=False)

duplicate_rows = int(df.duplicated().sum())
print("Duplicate rows:", duplicate_rows)
print("Numeric columns:", df.select_dtypes(include=np.number).columns.tolist())


## 5. Exploratory data analysis — sensor distributions

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
if numeric_df.empty:
    print("No numeric columns available for distribution plots.")
else:
    # Plot up to 12 numeric signals to keep the notebook readable.
    plot_cols = numeric_df.columns[:12]
    for col in plot_cols:
        series = numeric_df[col].replace([np.inf, -np.inf], np.nan).dropna()
        if series.empty:
            continue
        plt.figure(figsize=(8, 3))
        plt.hist(series, bins=50)
        plt.title(f"Distribution: {col}")
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.tight_layout()
        plt.savefig(REPORT_DIR / f"distribution_{col[:50]}.png", dpi=140)
        plt.show()


## 6. Flight trajectory visualization

In [ ]:
# Common coordinate naming patterns. Adjust these lists if the project uses different names.
latitude_candidates = ["latitude", "lat", "gps_lat", "lat_deg"]
longitude_candidates = ["longitude", "lon", "lng", "gps_lon", "lon_deg"]

def find_column(candidates, columns):
    lookup = {str(c).lower(): c for c in columns}
    for candidate in candidates:
        if candidate in lookup:
            return lookup[candidate]
    return None

lat_col = find_column(latitude_candidates, df.columns)
lon_col = find_column(longitude_candidates, df.columns)

if lat_col and lon_col:
    trajectory = df[[lat_col, lon_col]].apply(pd.to_numeric, errors="coerce").dropna()
    if not trajectory.empty:
        plt.figure(figsize=(7, 7))
        plt.plot(trajectory[lon_col], trajectory[lat_col], linewidth=0.8)
        plt.scatter(trajectory[lon_col].iloc[0], trajectory[lat_col].iloc[0], label="Start", s=35)
        plt.scatter(trajectory[lon_col].iloc[-1], trajectory[lat_col].iloc[-1], label="End", s=35)
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.title("Flight trajectory")
        plt.legend()
        plt.axis("equal")
        plt.tight_layout()
        plt.savefig(REPORT_DIR / "flight_trajectory.png", dpi=160)
        plt.show()
    else:
        print("Coordinate columns were found, but contain no valid coordinate pairs.")
else:
    print("No recognized latitude/longitude pair in this file. Update candidate names if needed.")


## 7. Prepare features for anomaly detection

This is a baseline on the selected processed file. It excludes likely identifiers/time fields by default. Review `DROP_COLUMNS` before training; do not include target labels, flight IDs, or future information as model features.

In [ ]:
# Add project-specific identifiers, timestamps, labels, or leakage columns here.
DROP_COLUMNS = {
    "timestamp", "time", "time_s", "flight_id", "window_id",
    "label", "anomaly", "is_anomaly", "remaining_flight_time",
    "remaining_time", "rul"
}

feature_df = df.select_dtypes(include=np.number).copy()
feature_df = feature_df.drop(
    columns=[c for c in feature_df.columns if str(c).lower() in DROP_COLUMNS],
    errors="ignore"
)
feature_df = feature_df.replace([np.inf, -np.inf], np.nan)
feature_df = feature_df.dropna(axis=1, how="all")

# Median imputation is fitted on this file only. For formal experiments,
# fit imputation separately on training data and apply it to validation/test.
feature_df = feature_df.fillna(feature_df.median(numeric_only=True))
feature_df = feature_df.loc[:, feature_df.nunique(dropna=True) > 1]

print("Feature matrix shape:", feature_df.shape)
print("Features:", feature_df.columns.tolist())
if feature_df.shape[1] == 0:
    print("No usable numeric features. Review DROP_COLUMNS and the selected dataset.")


## 8. Isolation Forest anomaly-detection baseline

In [ ]:
from sklearn.ensemble import IsolationForest

if feature_df.shape[1] == 0 or len(feature_df) < 20:
    print("Not enough usable rows/features to train Isolation Forest (need at least 20 rows and one feature).")
else:
    contamination = 0.02  # Initial assumption; tune using validation data/domain review.
    isolation_forest = IsolationForest(
        n_estimators=200,
        contamination=contamination,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    isolation_forest.fit(feature_df)

    # sklearn: -1 = anomaly, +1 = inlier
    predictions = isolation_forest.predict(feature_df)
    anomaly_scores = -isolation_forest.score_samples(feature_df)

    anomaly_results = pd.DataFrame({
        "row_index": feature_df.index,
        "prediction": predictions,
        "anomaly_score": anomaly_scores
    })
    anomaly_results["is_anomaly"] = anomaly_results["prediction"].eq(-1)

    anomaly_rate = float(anomaly_results["is_anomaly"].mean())
    print(f"Detected anomalies: {anomaly_results['is_anomaly'].sum()} / {len(anomaly_results)}")
    print(f"Anomaly fraction: {anomaly_rate:.2%}")

    display(anomaly_results.sort_values("anomaly_score", ascending=False).head(10))
    anomaly_results.to_csv(REPORT_DIR / "isolation_forest_predictions.csv", index=False)

    plt.figure(figsize=(9, 3))
    plt.plot(anomaly_results["anomaly_score"].to_numpy(), linewidth=0.7)
    plt.title("Isolation Forest anomaly scores")
    plt.xlabel("Row order")
    plt.ylabel("Anomaly score (higher = more unusual)")
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "isolation_forest_scores.png", dpi=160)
    plt.show()


## 9. Remaining-flight-time / RUL regression (conditional)

Only run this section if the data contains a trustworthy target column representing remaining flight time or remaining useful life. A target must be defined from flight/end-of-flight information without using future features as inputs.

If there is no valid target, this section deliberately skips training. Do not substitute an arbitrary sensor column and call it remaining flight time.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET_CANDIDATES = [
    "remaining_flight_time", "remaining_time_s",
    "remaining_time", "rul", "remaining_useful_life"
]
target_col = find_column(TARGET_CANDIDATES, df.columns)

if target_col is None:
    print("No recognized remaining-time/RUL target found. Regression is skipped.")
else:
    y = pd.to_numeric(df[target_col], errors="coerce")
    X = df.select_dtypes(include=np.number).drop(columns=[target_col], errors="ignore")
    X = X.drop(columns=[c for c in X.columns if str(c).lower() in DROP_COLUMNS], errors="ignore")
    X = X.replace([np.inf, -np.inf], np.nan)
    valid = y.notna()
    X, y = X.loc[valid], y.loc[valid]
    X = X.fillna(X.median(numeric_only=True))
    X = X.loc[:, X.nunique(dropna=True) > 1]

    if len(X) < 30 or X.shape[1] == 0:
        print("Insufficient valid rows/features for a meaningful regression baseline.")
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=RANDOM_STATE
        )
        regressor = RandomForestRegressor(
            n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
        )
        regressor.fit(X_train, y_train)
        y_pred = regressor.predict(X_test)

        metrics = {
            "target": target_col,
            "mae": float(mean_absolute_error(y_test, y_pred)),
            "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
            "r2": float(r2_score(y_test, y_pred))
        }
        print(json.dumps(metrics, indent=2))
        with open(REPORT_DIR / "rul_baseline_metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)


## 10. Save a reproducibility summary

In [ ]:
summary = {
    "project_root": str(PROJECT_ROOT.resolve()),
    "selected_file": str(selected_path.relative_to(PROJECT_ROOT)),
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "numeric_feature_count": int(feature_df.shape[1]),
    "random_state": RANDOM_STATE,
    "outputs_directory": str(REPORT_DIR.relative_to(PROJECT_ROOT))
}
with open(REPORT_DIR / "notebook_run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nNotebook outputs saved to:", REPORT_DIR)


## 11. Research checklist and next implementation steps

- [ ] Validate PX4 ULog and INAV ingestion independently.
- [ ] Confirm timestamp units, sampling rates, sensor units, and synchronization.
- [ ] Generate flight-level and window-level train/validation/test splits before fitting preprocessing.
- [ ] Compare Isolation Forest with the existing LSTM Autoencoder using the same held-out flights.
- [ ] Implement PatchTST and Anomaly Transformer; record configuration, seeds, runtime, and metrics.
- [ ] Define a defensible energy-aware remaining-flight-time target and evaluate by flight, not random row split.
- [ ] Evaluate cross-hardware generalization (train on one hardware family, test on the other).
- [ ] Add SHAP explanations for supported models and attention maps for transformer models.
- [ ] Export the selected model to ONNX and measure latency/size before and after INT8 quantization.
- [ ] Connect the validated model to FastAPI, Docker, and MLflow model registry.
- [ ] Add automated tests and GitHub Actions checks.

**Interpretation note:** The plots and baseline scores from one selected file are exploratory only. They are not final scientific results until repeated across flight-level splits and documented with suitable metrics and uncertainty.
